<a href="https://colab.research.google.com/github/audomsak-pypy/Project1-Text_Analytics69/blob/Toto/Project1_Text_Analytics69_Teams_673020271_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project: Text Analytics for Business Insight
### SC 663 402 Data Warehouse and Big Data Analytics

## โครงสร้างงานและการส่งงาน

งานนี้แบ่งเป็น **งานเดี่ยว 1 ส่วน** และ **งานกลุ่ม 2 ส่วน**

### งานเดี่ยว — Part 1 (10 คะแนน)
* **นักศึกษาทุกคนต้องทำ Part 1 ด้วยตนเองและส่งของตนเองทุกคน**
* ให้ส่ง Notebook ของตนเองพร้อมผลรันครบ ไม่ใช่ใช้ไฟล์เดียวกันทั้งกลุ่ม
* สามารถพูดคุยแนวคิดหรือช่วยกันแก้ปัญหาทั่วไปได้ แต่โค้ด ผลวิเคราะห์ และคำตอบที่ส่งต้องเป็นงานของตนเอง
* Part 1 **ไม่ต้องนำเสนอหน้าชั้นเรียน**

### งานกลุ่ม — Part 2 + Part 3 (75 คะแนน) และการนำเสนอ (15 คะแนน)
* Part 2 และ Part 3 เป็น **งานกลุ่ม** และ **ส่งเพียง 1 คนต่อทีม**
* ในหน้าแรกของงานกลุ่มต้องระบุว่าใครรับผิดชอบส่วนใดอย่างชัดเจน
* โดยปกติให้รวม Part 2 และ Part 3 ไว้ใน Notebook เดียว
* หาก Notebook มีปัญหา RAM/หน่วยความจำ หรือไฟล์มีขนาดใหญ่เกินไป **อนุญาตให้แยกงานกลุ่มเป็น 2 Notebook** เช่น `TeamXX_Part2.ipynb` และ `TeamXX_Part3.ipynb` ได้ โดยแต่ละไฟล์ต้องมี setup และคำอธิบาย path ที่ทำให้เปิดแล้วรันต่อได้
* ไม่ว่าจะแยกไฟล์หรือไม่ ให้ถือว่าเป็นผลงานกลุ่มชุดเดียว และส่งโดยตัวแทนทีมเพียงคนเดียว

> **การประเมินรายบุคคลในงานกลุ่ม:** คะแนนผลงาน Part 2–3 เป็นคะแนนทีม แต่คะแนนการตอบคำถามในวันนำเสนอจะพิจารณาเป็นรายบุคคล โดยเฉพาะในส่วนที่สมาชิกคนนั้นระบุว่าเป็นผู้รับผิดชอบ


In [ ]:
"""
หากยังไม่มี library ใด ให้ติดตั้งก่อน
!pip install -q nltk wordcloud scikit-learn pandas matplotlib tqdm huggingface_hub
!pip install -q pythainlp          # ใช้เมื่อทำงานกับข้อความภาษาไทย
"""
import json, os, re, random, itertools, time
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nltk
for pkg in ["punkt_tab", "stopwords", "wordnet", "vader_lexicon"]:
    nltk.download(pkg, quiet=True)

%matplotlib inline

RANDOM_SEED = 42
SAMPLE_N = 20_000         # เวอร์ชันลด workload: Part 2 ใช้อย่างน้อย 20,000 รีวิว
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

DATA_DIR = "./data/"
os.makedirs(DATA_DIR, exist_ok=True)

---
# Part 1: Social Listening Warm-up
*งานเดี่ยว — ทุกคนต้องทำและส่งของตนเอง ไม่ต้องนำเสนอ (10 คะแนน)*

ใช้ข้อมูลโพสต์สาธารณะจาก **Bluesky**  
https://huggingface.co/datasets/alpindale/two-million-bluesky-posts

ใช้เพียง **1–3 ไฟล์** ก็พอ ไม่จำเป็นต้องโหลดทั้ง dataset

ฟิลด์ที่มี เช่น `text`, `created_at`, `author`, `uri`, `has_images`, `reply_to`, `predicted_language`

## ตอบ 4 คำถามหลัก

1. **คนโพสต์เมื่อไร:** จำนวนโพสต์ ช่วงเวลาที่ข้อมูลครอบคลุม และชั่วโมงที่โพสต์มากที่สุด
2. **ใช้ภาษาอะไร:** รายงานภาษาหลักและสัดส่วน เพื่อบอกว่าถ้าทำคอนเทนต์ควรรองรับภาษาใด
3. **คนพูดถึงอะไร:** หา hashtag หรือคำ/วลีเด่นที่น่าสนใจ และยกตัวอย่างข้อความจริงประกอบ
4. **ข้อมูลนี้บอกอะไรได้และบอกอะไรไม่ได้:** สร้าง visualization 1 รูป แล้วเขียนข้อจำกัด 3–5 บรรทัด

> Part 1 เป็น warm-up ไม่ต้องทำ influencer analysis หรือ reply-network เว้นแต่สนใจทำเพิ่มเอง


In [ ]:
# ----------------- Your code here -----------------
# ใบ้: อ่าน JSON lines ทีละบรรทัดด้วย generator ไม่ต้องโหลดทั้งไฟล์
# def iter_jsonl(path):
#     with open(path, encoding='utf-8') as f:
#         for line in f:
#             yield json.loads(line)

In [ ]:
# ----------------- Your code here -----------------

**<สรุปสั้น ๆ ถึงลูกค้า 5 บรรทัด>**

* ...